# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [28]:
# Importation des packages nécessaires

import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [29]:
# Chargement des données
df_mapping_initial = pd.read_csv("../data_finale/mapping_fbref_tm.csv", encoding='latin1')
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [30]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [31]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm)

[1] Nom exact (mapping)     : 17067 | restants : 833
[2] Fuzzy nom (mapping)     :    55 | restants : 778


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [32]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')

In [33]:
df_final

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Performance_GA,Performance_GA90,Performance_SoTA,Performance_Saves,Performance_Save%,Performance_W,Performance_D,Performance_L,Performance_CS,Performance_CS%,Penalty Kicks_PKatt,Penalty Kicks_PKA,Penalty Kicks_PKsv,Penalty Kicks_PKm,Penalty Kicks_Save%,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year,tm_id,match_method,player_id,valuation_year,market_value_in_eur,date,date_of_birth,name,tm_join_key,tm_join_key_full,tm_dob_key
0,ENG-Premier League,2021,Arsenal,Ainsley Maitland-Niles,ENG,"MF,DF",22,1997.0,11,5,490,5.4,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,0,2,1,50.0,0.37,0.18,0.00,0.00,0,0,45,14.3,5,85.0,4,6,11.0,8,1.64,8,6,2,0.37,-0.06,0,6,3,2,8,14,8,NaN,NaN,0,0.796191,0.821991,0.796191,5.968475,5.051885,ainsley maitland niles,1997.0,2021,285845.0,exact_name_mapping,285845.0,2021.0,12000000.0,2021-12-23,1997-08-29 00:00:00,Ainsley Maitland-Niles,ainsley maitland niles,ainsley maitland niles,1997
1,ENG-Premier League,2021,Arsenal,Alexandre Lacazette,FRA,FW,29,1991.0,31,22,1923,21.4,13,2,15,10,3,3,3,0,0.61,0.09,0.70,0.47,0.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.4,13,45,29,64.4,2.11,1.36,0.22,0.34,3,3,62,56.2,22,80.0,7,9,19.0,5,1.52,32,21,11,0.51,0.21,0,37,54,10,3,10,9,NaN,NaN,0,12.028911,2.209213,9.745403,14.361544,2.937946,alexandre lacazette,1991.0,2021,93720.0,exact_name_mapping,93720.0,2021.0,20000000.0,2021-12-23,1991-05-28 00:00:00,Alexandre Lacazette,alexandre lacazette,alexandre lacazette,1991
2,ENG-Premier League,2021,Arsenal,Bernd Leno,GER,GK,28,1992.0,35,35,3131,34.8,0,0,0,0,0,0,0,1,0.00,0.00,0.00,0.00,0.00,37.0,1.06,120.0,84.0,70.0,17.0,6.0,12.0,11.0,31.4,2.0,1.0,1.0,0.0,50.0,34.8,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,89,91.5,35,89.0,34,0,NaN,2,1.63,52,37,15,0.43,0.12,0,2,4,0,0,0,1,NaN,NaN,1,0.000000,0.000000,0.000000,4.326598,4.326598,bernd leno,1992.0,2021,72476.0,exact_name_mapping,72476.0,2021.0,16000000.0,2021-12-23,1992-03-04 00:00:00,Bernd Leno,bernd leno,bernd leno,1992
3,ENG-Premier League,2021,Arsenal,Bukayo Saka,ENG,MF,18,2001.0,32,30,2553,28.4,5,3,8,5,0,0,1,0,0.18,0.11,0.28,0.18,0.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.4,5,61,20,32.8,2.15,0.71,0.08,0.25,0,0,80,74.6,30,83.0,20,2,25.0,4,1.59,38,31,7,0.25,-0.69,0,28,64,15,147,26,16,NaN,NaN,1,7.169173,4.511791,7.169173,18.292467,9.180444,bukayo saka,2001.0,2021,433177.0,exact_name_mapping,433177.0,2021.0,65000000.0,2021-12-23,2001-09-05 00:00:00,Bukayo Saka,bukayo saka,bukayo saka,2001
4,ENG-Premier League,2021,Arsenal,Calum Chambers,ENG,DF,25,1995.0,10,8,753,8.4,0,2,2,0,0,0,0,0,0.00,0.24,0.24,0.00,0.24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.4,0,8,0,0.0,0.96,0.00,0.00,NaN,0,0,75,22.0,8,89.0,7,2,21.0,8,2.00,16,10,6,0.72,0.38,0,8,2,0,33,10,3,NaN,NaN,0,0.363973,1.042764,0.363973,3.835656,2.810006,calum chambers,1995.0,2021,215118.0,exact_name_mapping,215118.0,2021.0,12000000.0,2021-12-23,1995-01-20 00:00:00,Calum Chambers,calum chambers,calum chambers,1995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

## **Les joueurs orphelins**

In [25]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 685
Taux joueurs orphelins : 11.06%


In [26]:
orphelins_par_saison = (
    still_missing
    .groupby('season')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season
2021     12
2122     36
2223     48
2324     81
2425     49
2526    552
dtype: int64


In [27]:
top_orphelins = (
    still_missing['player']
    .value_counts()
    .head(20)
)

print(top_orphelins)

player
Babis Lykogiannis    6
Hianga'a Mbock       5
Jonathan Rowe        4
Álex Jiménez         4
Yellu Santiago       4
Karl Etta Eyong      4
Ben Gannon-Doak      3
Kosta Nedeljković    3
Manuel Ugarte        3
Benjamin Šeško       3
Marc Pubill          3
Abde Rebbach         3
Bertuğ Yıldırım      3
Abdulai Juma Bah     3
Ismaël Boura         3
Hákon Haraldsson     3
Merveille Papela     3
Keke Topp            3
Dženan Pejčinović    3
Domen Črnigoj        3
Name: count, dtype: int64
